In [ ]:
from pyscf import gto, scf, lib
import numpy as np
from pyscf.hessian import rhf as rhf_hess
from pyscf.df.hessian import rhf as df_rhf_hess

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
mf = scf.RHF(mol).density_fit()
mf.with_df.build()
mf.run()

converged SCF energy = -56.1132387662623


## 参考值计算

计算 RI-JK RHF Hessian 的参考值 `de_ref`。

该分解的目标是：

$$\text{de\_ref} = \text{de\_hcore} + \text{de\_ovlp} + \text{de\_J\_(basis\_2nd)} + \text{de\_J\_(basis\_1st\_aux\_1st)} + \text{de\_J\_(aux\_2nd)} - \text{de\_K\_(basis\_2nd)} - \text{de\_K\_(basis\_1st\_aux\_1st)} - \text{de\_K\_(aux\_2nd)} + \text{de\_cphf} + \text{de\_nuc}$$

其中各贡献按 PySCF 的 notation 分类：
- **basis\_2nd**: 全部导数在轨道基上 (0 在辅助基)，包括 $(20|0)(0|00)$, $(11|0)(0|00)$, $(10|0)(0|10)$
- **basis\_1st\_aux\_1st**: 轨道基 1阶 + 辅助基 1阶，包括 $(10|1)(0|00)$, $(10|0)(0|1)(0|00)$, $(10|0)(1|0)(0|00)$, $(10|0)(1|00)$ 等
- **aux\_2nd**: 全部导数在辅助基上 (0 在轨道基)，包括 $(00|2)(0|00)$, $(00|0)(1|1)(0|00)$ 等

In [4]:
mf_hess = mf.Hessian().run()
de_ref = mf_hess.de.copy()
print("de_ref shape:", de_ref.shape)

de_ref shape: (4, 4, 3, 3)


## 基本量提取

In [5]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mo_energy = mf.mo_energy
nao, nmo = mo_coeff.shape
mocc = mo_coeff[:, mo_occ > 0]
mocc_2 = np.einsum('pi,i->pi', mocc, mo_occ[mo_occ > 0]**.5)
nocc = mocc.shape[1]
dm0 = np.dot(mocc, mocc.T) * 2
dme0 = np.einsum('pi,qi,i->pq', mocc, mocc, mo_energy[mo_occ > 0]) * 2
natm = mol.natm
atmlst = range(natm)
aoslices = mol.aoslice_by_atom()

# 基本分解

## 核排斥贡献

In [6]:
de_nuc = rhf_hess.hess_nuc(mol)

## `_partial_hess_ejk` 分解

利用 `auxbasis_response` 参数的不同取值来提取不同阶数的辅助基贡献：

- `auxbasis_response = 0`: 只计算轨道基导数 → **basis\_2nd** 贡献
- `auxbasis_response = 1`: 添加一阶辅助基响应 (J factor 1.0, K factor 0.5)
- `auxbasis_response = 2`: 添加完整一阶 + 二阶辅助基响应 (J factor 2.0, K factor 1.0)

分解公式：
- `J_basis_2nd = ej_aux0` (aux0 结果)
- `J_basis_1st_aux_1st = 2 × (ej_aux1 - ej_aux0)` (修正为 full factor 2.0)
- `J_aux_2nd = ej_aux2 - 2×ej_aux1 + ej_aux0`
- K 同理

In [7]:
# auxbasis_response = 0: only orbital derivatives (basis_2nd)
hessobj_aux0 = mf.Hessian()
hessobj_aux0.auxbasis_response = 0
e1_aux0, ej_aux0, ek_aux0 = df_rhf_hess._partial_hess_ejk(hessobj_aux0)

In [8]:
# auxbasis_response = 1: 1st-order aux response (J factor 1.0, K factor 0.5)
hessobj_aux1 = mf.Hessian()
hessobj_aux1.auxbasis_response = 1
e1_aux1, ej_aux1, ek_aux1 = df_rhf_hess._partial_hess_ejk(hessobj_aux1)

In [9]:
# auxbasis_response = 2: full aux response (default, J factor 2.0, K factor 1.0)
hessobj_aux2 = mf.Hessian()
hessobj_aux2.auxbasis_response = 2
e1_aux2, ej_aux2, ek_aux2 = df_rhf_hess._partial_hess_ejk(hessobj_aux2)

## e1 分解 (hcore + overlap)

`e1` 包含 core Hamiltonian 二阶导数与 overlap 二阶导数两部分：

- **de\_hcore**: $\sum_{A,B} \langle \nabla_A \nabla_B H_{\text{core}} | D_0 \rangle$
- **de\_ovlp**: $-\sum_{A,B} \langle \nabla_A \nabla_B S | D_{E,0} \rangle$ (能量加权密度矩阵)

e1 对所有 `auxbasis_response` 值相同 (不含辅助基响应)。

In [10]:
# e1 is the same for all auxbasis_response levels (no aux dependence)
e1 = e1_aux2.copy()

hcore_deriv = mf_hess.hcore_generator(mol)
s1aa, s1ab, s1a_ovlp = rhf_hess.get_ovlp(mol)

de_hcore = np.zeros((natm, natm, 3, 3))
de_ovlp = np.zeros((natm, natm, 3, 3))

for i0, ia in enumerate(atmlst):
    shl0, shl1, p0, p1 = aoslices[ia]
    # overlap diagonal: s1aa contracted with dme0
    de_ovlp[i0, i0] -= np.einsum('xypq,pq->xy', s1aa[:, :, p0:p1], dme0[p0:p1]) * 2
    for j0, ja in enumerate(atmlst[:i0 + 1]):
        q0, q1 = aoslices[ja][2:]
        # hcore second derivative contracted with dm0
        h1ao_hc = hcore_deriv(ia, ja)
        de_hcore[i0, j0] += np.einsum('xypq,pq->xy', h1ao_hc, dm0)
        # overlap cross: s1ab contracted with dme0
        de_ovlp[i0, j0] -= np.einsum('xypq,pq->xy', s1ab[:, :, p0:p1, q0:q1], dme0[p0:p1, q0:q1]) * 2
    for j0 in range(i0):
        de_hcore[j0, i0] = de_hcore[i0, j0].T
        de_ovlp[j0, i0] = de_ovlp[i0, j0].T

print("e1 = hcore + ovlp:", np.allclose(e1, de_hcore + de_ovlp))

e1 = hcore + ovlp: True


## J 贡献分解

In [ ]:
# J_basis_2nd = ej with auxbasis_response = 0 (orbital-only derivatives)
# This includes (20|0)(0|00), (11|0)(0|00), (10|0)(0|10) contributions
de_J_20 = ej_aux0.copy()

# J_basis_1st_aux_1st: full 1st-order aux response with correct factor 2.0
# aux1 gives factor 1.0, aux2 gives factor 2.0, so we scale the difference by 2
de_J_11 = 2.0 * (ej_aux1 - ej_aux0)

# J_aux_2nd: 2nd-order aux response
de_J_02 = ej_aux2 - 2.0 * ej_aux1 + ej_aux0

print("J decomposition check:", np.allclose(ej_aux2, de_J_20 + de_J_11 + de_J_02))

J decomposition check: True


## K 贡献分解

In [ ]:
# K_basis_2nd = ek with auxbasis_response = 0 (orbital-only derivatives)
de_K_20 = ek_aux0.copy()

# K_basis_1st_aux_1st: full 1st-order aux response with correct factor 1.0
# aux1 gives factor 0.5, aux2 gives factor 1.0, so we scale the difference by 2
de_K_11 = 2.0 * (ek_aux1 - ek_aux0)

# K_aux_2nd: 2nd-order aux response
de_K_02 = ek_aux2 - 2.0 * ek_aux1 + ek_aux0

print("K decomposition check:", np.allclose(ek_aux2, de_K_20 + de_K_11 + de_K_02))

K decomposition check: True


## CPHF 响应

In [13]:
# Compute full hess_elec = partial_hess_elec + CPHF response
de_hess_elec = mf_hess.hess_elec()

# CPHF response = hess_elec - partial_hess_elec
de_partial = e1 + ej_aux2 - ek_aux2
de_cphf = de_hess_elec - de_partial

print("partial_hess_elec check:", np.allclose(de_partial, mf_hess.partial_hess_elec()))
print("hess_elec = partial + cphf:", np.allclose(de_hess_elec, de_partial + de_cphf))

partial_hess_elec check: True
hess_elec = partial + cphf: True


## 总核验

In [ ]:
de_sum = de_hcore + de_ovlp \
         + de_J_20 + de_J_11 + de_J_02 \
         - de_K_20 - de_K_11 - de_K_02 \
         + de_cphf + de_nuc

print("de_ref == de_sum:", np.allclose(de_ref, de_sum))
print("max abs difference:", np.max(np.abs(de_ref - de_sum)))

de_ref == de_sum: True
max abs difference: 2.293165657363261e-13


In [ ]:
print("========== Contribution Summary ==========")
contributions = {
    "hcore":               de_hcore,
    "ovlp":                de_ovlp,
    "J_basis_2nd":         de_J_20,
    "J_basis_1st_aux_1st": de_J_11,
    "J_aux_2nd":           de_J_02,
    "K_basis_2nd":         de_K_20,
    "K_basis_1st_aux_1st": de_K_11,
    "K_aux_2nd":           de_K_02,
    "cphf":                de_cphf,
    "nuc":                 de_nuc,
}

for name, arr in contributions.items():
    sign = "+" if name.startswith("J") or name in ["hcore", "ovlp", "cphf", "nuc"] else "-"
    print(f"  {sign} {name:30s}: max = {np.max(np.abs(arr)):12.6f}, norm = {np.linalg.norm(arr):12.6f}")

print(f"\n  {'de_ref':30s}: max = {np.max(np.abs(de_ref)):12.6f}, norm = {np.linalg.norm(de_ref):12.6f}")

========== Contribution Summary ==========
  + hcore                         : max =     4.796595, norm =    15.424534
  + ovlp                          : max =     0.471350, norm =     1.219724
  + J_basis_2nd                   : max =    26.681874, norm =    46.496180
  + J_basis_1st_aux_1st           : max =    44.752165, norm =    77.479630
  + J_aux_2nd                     : max =    22.376444, norm =    38.740809
  - K_basis_2nd                   : max =    11.977857, norm =    20.655346
  - K_basis_1st_aux_1st           : max =    22.918044, norm =    39.671048
  - K_aux_2nd                     : max =    11.459257, norm =    19.835893
  + cphf                          : max =     0.257821, norm =     0.743761
  + nuc                           : max =     1.810204, norm =     5.912849

  de_ref                        : max =     0.478794, norm =     1.008162


# 详细分解